# Patent Landscaping — Snorkel vs MAS (expanded + unified)

**Final experiment.** Both arms label the **full candidate set** (`candidate_all` = expanded
autonomous pool 24,488 + OOD 6,296). MAS labels are committed (`mas_ranked_scores.csv`);
Snorkel labels the same set here. Then we fine-tune SciBERT for each arm under two negative
mixes: **all-OOD** (baseline) vs **in-domain (equal-N)**.

Runtime → GPU (T4) → Run all. Models persist to Drive (restore in later sessions, no retrain).

In [ ]:
# 1) setup
import os
%cd /content
REPO = 'https://github.com/PassionChicken-Leesuin/Patent_Landscaping_Final.git'
if not os.path.exists('/content/Patent_Landscaping_Final'):
    !git clone $REPO
%cd /content/Patent_Landscaping_Final
!git pull
!pip -q install snorkel transformers datasets scikit-learn accelerate

In [ ]:
# 2) mount Drive (persistent model store)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/patent_landscaping_outputs'
os.makedirs(DRIVE, exist_ok=True); os.makedirs('outputs', exist_ok=True)
print('Drive store:', DRIVE)

In [ ]:
# 3) data prep — processed sets + build the unified candidate set (expanded pool + OOD)
!python -m scripts.build_dataset
!python -m scripts.build_candidate_all

In [ ]:
# 4) SNORKEL — label the FULL candidate set (autonomous pool + OOD together)
!python -m scripts.run_snorkel --input DataSet/processed/candidate_all.csv

MAS already labeled `candidate_all` (committed `mas_ranked_scores.csv`, 30,784 rows) — no API call here.

In [ ]:
# 5) train: each arm x {all-OOD, equal-N}.  Restore from Drive if present, else train + save.
import shutil
FORCE = False
EPOCHS, MAX_LEN = 4, 256

def ensure(name, run_args):
    local, dstore = f'outputs/scibert_{name}', f'{DRIVE}/scibert_{name}'
    if not FORCE and os.path.isfile(f'{local}/config.json'):
        if not os.path.isfile(f'{dstore}/config.json'):
            shutil.copytree(local, dstore, dirs_exist_ok=True)
            if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
        print(f'[{name}] local'); return
    if not FORCE and os.path.isfile(f'{dstore}/config.json'):
        shutil.copytree(dstore, local, dirs_exist_ok=True)
        if os.path.isfile(f'{DRIVE}/metrics_{name}.json'): shutil.copy(f'{DRIVE}/metrics_{name}.json', 'outputs/')
        print(f'[{name}] restored'); return
    print(f'[{name}] training (~30 min) ...')
    get_ipython().system(f'python -m scripts.run_downstream {run_args} --epochs {EPOCHS} --max-len {MAX_LEN}')
    shutil.copytree(local, dstore, dirs_exist_ok=True)
    if os.path.isfile(f'outputs/metrics_{name}.json'): shutil.copy(f'outputs/metrics_{name}.json', DRIVE)
    print(f'[{name}] done + saved')

# equal-N (in-domain negatives, balanced) — the key configs
ensure('mas_uni_equalN',     '--arm mas --unified --neg-pos-ratio 1.0 --tag uni_equalN')
ensure('snorkel_uni_equalN', '--arm snorkel --unified --neg-pos-ratio 1.0 --tag uni_equalN')
# all-OOD (baseline negative mix) — contrast
ensure('mas_uni_allood',     '--arm mas --unified --tag uni_allood')
ensure('snorkel_uni_allood', '--arm snorkel --unified --tag uni_allood')

In [ ]:
# 6) train/val curves
from src.downstream.plots import plot_history
for n in ['mas_uni_equalN', 'snorkel_uni_equalN']:
    try: plot_history(n)
    except FileNotFoundError: print(f'[{n}] no history')

In [ ]:
# 7) compare on gold eval (@0.5). Watch SEED-recall: expanded pool + in-domain negs should lift it.
import json
def show(name):
    p = f'outputs/metrics_{name}.json'
    if not os.path.isfile(p): print(f'{name:20s} (missing)'); return
    r = json.load(open(p))
    seed = r['by_expansion_level'].get('SEED', {}).get('recall(TP rate)', float('nan'))
    print(f"{name:20s} F1={r['macro_f1']:.3f} AUC={r['auc']:.3f} "
          f"AP={r.get('average_precision', float('nan')):.3f} R={r['recall']:.3f} "
          f"P={r['precision']:.3f} | SEED-recall={seed:.3f}")
for n in ['snorkel_uni_allood', 'snorkel_uni_equalN', 'mas_uni_allood', 'mas_uni_equalN']:
    show(n)